# Fase 2f: CT 2.5D mixed-context training

Este experimento usa slices adyacentes como canales de entrada: `z-1`, `z` y `z+1`. La hipotesis es que la segmentacion CT de infeccion mejora cuando el modelo ve contexto volumetrico local, manteniendo validacion y test sobre slices completos 256x256.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import config
from src.data.segmentation import (
    build_ct_segmentation_dataframe,
    split_segmentation_dataframe,
    SegmentationDataset,
    SegmentationPairTransform,
)
from src.training.segmentation_experiment import (
    get_device,
    make_segmentation_run_config,
    train_and_evaluate_segmentation,
    save_segmentation_artifacts,
    existing_segmentation_artifact,
    summarize_segmentation_results,
)

# En macOS/Jupyter es mas estable para este dataset pequeno.
config.NUM_WORKERS = 0

device = get_device()
print(f'Device: {device}')

## 2. Configuracion experimental

In [ ]:
run_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode='full',
    image_size=config.CT_IMAGE_SIZE,
    in_channels=3,
    variant_name='ct25d_mixed50_patch160_pos80_tversky_pos20_thr',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=20.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    train_crop_size=(160, 160),
    train_crop_prob=0.5,
    positive_crop_prob=0.8,
    batch_size=8,
    epochs=18,
    early_stopping_patience=5,
    base_features=16,
    ct_context_slices=True,
)
run_config

## 3. Datos, splits y disponibilidad de contexto

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)

train_df, val_df, test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

print({'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print({'train_studies': train_df.study_id.nunique(), 'val_studies': val_df.study_id.nunique(), 'test_studies': test_df.study_id.nunique()})

def context_availability(df):
    total = len(df)
    prev_count = 0
    next_count = 0
    both_count = 0
    for row in df.itertuples():
        image_path = Path(row.image_path)
        prev_path = image_path.with_name(f'{row.study_id}_slice_{int(row.slice_index) - 1:03d}.png')
        next_path = image_path.with_name(f'{row.study_id}_slice_{int(row.slice_index) + 1:03d}.png')
        has_prev = prev_path.exists()
        has_next = next_path.exists()
        prev_count += int(has_prev)
        next_count += int(has_next)
        both_count += int(has_prev and has_next)
    return {
        'prev_available': prev_count / total,
        'next_available': next_count / total,
        'both_available': both_count / total,
    }

pd.DataFrame([
    {'split': 'train', **context_availability(train_df)},
    {'split': 'val', **context_availability(val_df)},
    {'split': 'test', **context_availability(test_df)},
])

## 4. Comprobacion de tensores

In [ ]:
sample_transform = SegmentationPairTransform(config.CT_IMAGE_SIZE, in_channels=3, is_train=False)
sample_dataset = SegmentationDataset(
    test_df.head(2),
    transform=sample_transform,
    in_channels=3,
    ct_context_slices=True,
)
image, mask = sample_dataset[0]
print('image:', tuple(image.shape), 'mask:', tuple(mask.shape), 'min/max:', float(image.min()), float(image.max()))

## 5. Entrenamiento

In [ ]:
seg_models_dir = config.MODELS_DIR / 'segmentation' / 'ct'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation' / 'ct'

if existing_segmentation_artifact(run_config, seg_models_dir, seg_results_dir):
    print(f'Saltado: artefactos existentes detectados para {run_config.experiment_name}_{run_config.run_mode}.')
else:
    result = train_and_evaluate_segmentation(run_config, train_df, val_df, test_df, device)
    artifact_paths = save_segmentation_artifacts(run_config, result, seg_models_dir, seg_results_dir)
    print(artifact_paths)
    print(result['metrics'])
    print('threshold:', result['threshold'])

## 6. Comparacion CT

In [ ]:
summary_df = summarize_segmentation_results(sorted(seg_results_dir.glob('*_summary.json')))
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
summary_df[cols].sort_values('dice', ascending=False).reset_index(drop=True)

## 7. Hueco para variantes siguientes

Si 2.5D mejora, el siguiente barrido razonable es conservar mas contexto y reducir ponderacion positiva. No ejecutes todas las variantes sin comparar primero el resultado principal.

In [ ]:
NEXT_VARIANTS = [
    {
        'variant_name': 'ct25d_mixed30_patch192_pos70_tversky_pos10_thr',
        'train_crop_size': (192, 192),
        'train_crop_prob': 0.3,
        'positive_crop_prob': 0.7,
        'pos_weight': 10.0,
        'base_features': 16,
        'epochs': 18,
    },
    {
        'variant_name': 'ct25d_full_context_tversky_pos10_thr',
        'train_crop_size': None,
        'train_crop_prob': 0.0,
        'positive_crop_prob': 0.0,
        'pos_weight': 10.0,
        'base_features': 16,
        'epochs': 24,
    },
]
NEXT_VARIANTS